<!-- <style> * { font-size: 100%; } </style> -->
<small> 

# rsyncd
- https://linux.die.net/man/5/rsyncd.conf
- https://superuser.com/questions/243656/how-to-configure-and-use-rsyncd


</small>

In [ ]:
## Setup Working Directory & Files

# Working Dir
WORKING_DIRECTORY="/srv/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# Make Folders & Files
# mkdir -p ./{config,repositories}
# mkdir -p ./{config,secrets}
# ls -alt ${WORKING_DIRECTORY}

# Create Needed Files
# touch ${WORKING_DIRECTORY}/secrets/rsyncd.secrets

In [ ]:
# Generate User Passwords
printf 'username:%s\n' "$(openssl rand -base64 64 | head -c 64 | tr -d '\n' | tr -dc 'A-Za-z0-9')"

In [ ]:
# Generate Passwords
COUNT=12
LENGTH=64
FILTER='A-Za-z0-9' # alnum:'A-Za-z0-9'  # low:'a-z0-9'  # hex:'A-Fa-f0-9'

printf 'Password Generator | COUNT=%s | LENGTH=%s | FILTER=%s\n' \
  "$COUNT" "$LENGTH" "$FILTER"

for ((i=1; i<=COUNT; i++)); do
  openssl rand -base64 "$((LENGTH + 12))" | tr -dc 'A-Za-z0-9' | head -c "$LENGTH"
  echo
done

# Docker

## Docker Context

In [ ]:
# Use Context
# docker context use default
docker context use nas-03

## Build and Deploy Container

In [ ]:
# Build (example)
docker compose --progress plain --profile ssh --file ../docker-compose.yaml --env-file ../docker-compose.example.env build

In [ ]:
# Build w/ SSH
docker compose --progress plain --profile ssh --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env build

In [ ]:
# Build & Run (no ssh)
docker compose --progress plain --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

In [ ]:
# Build & Run (w/ SSH)
docker compose --progress plain --profile ssh --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

In [ ]:
# Build & Run in Context
CONTEXT=nas-03
# docker --context lxc-docker-01 compose --profile "*" --env-file .env.lxc-docker  up --build --detach --remove-orphans

# Build the Base Image for Minecraft Bedrock Server Backup (Bedrockifier)
docker --context "${CONTEXT}" build \
  --progress plain \
  --tag local/rsync-server:latest \
  --env-file ../docker-compose.nas-03_unraid.env up
  --file ../docker-compose.yaml \
  --profile ssh

# Build + Run the Minecraft Bedrock Server & Backup Service
docker --context "${CONTEXT}" compose \
  --ansi never \
  --env-file ../.env.lxc-docker \
  up --build --detach --remove-orphans

## Synology SSH Notes

- For Synology Hyper Backup, use `rsync-compatible server` with `Transfer encryption` set to `On`.
- The SSH flow sends `rsync --server --daemon .` over SSH, so the SSH container needs to serve rsync modules for the logged-in user.
- A good working pattern is one SSH user per backup module, for example `nas-01` -> `nas-01`.
- In Hyper Backup, `Backup module` should be the rsync module name such as `nas-01`. Leave `Directory` blank unless you intentionally want a subfolder under that module.
- If you set `Directory` to something like `NAS-01`, the backup lands under `/data/servers/nas-01/NAS-01`.


In [ ]:
# Test Synology-style rsync over SSH: list available modules for one SSH user
SSH_PORT=2222
SSH_USER=nas-01
SSH_HOST=nas-03.internal
rsync -e "ssh -p ${SSH_PORT}" ${SSH_USER}@${SSH_HOST}::


In [ ]:
# Test Synology-style rsync over SSH: list contents of a specific module
SSH_PORT=2222
SSH_USER=nas-01
SSH_HOST=nas-03.internal
MODULE_NAME=nas-01
rsync -e "ssh -p ${SSH_PORT}" ${SSH_USER}@${SSH_HOST}::${MODULE_NAME}


## Troubleshooting

- If SSH authentication fails, manually type the password in Hyper Backup and confirm it matches the user in `RSYNCD_USERS` or the active secrets file.
- If SSH login works but Hyper Backup cannot find modules, test the same flow manually with `rsync -e "ssh -p 2222" USER@HOST::`.
- If you see `rsync: did not see server greeting`, the SSH session is not successfully serving `rsync --server --daemon .` yet. Check `docker logs rsyncd-server-ssh`.
- The SSH container must expose modules for the logged-in SSH user. A simple working pattern is one SSH user per rsync module, for example `nas-01` -> `nas-01`.
- For Hyper Backup, the `Backup module` should be the module name such as `nas-01`. Leave `Directory` blank unless you intentionally want a subfolder under that module.
- During debugging, `RSYNC_SSH_TRACE_COMMANDS=true` and `RSYNC_SSH_LOG_LEVEL=DEBUG1` are useful. After things work, set tracing back to `false` and lower log level back to `INFO`.


In [ ]:
# SSH / Synology troubleshooting logs
docker logs --tail 200 rsyncd-server-ssh
docker logs --tail 200 rsyncd-server


The server side is working now. I tested with your real nas-01 credentials from this machine and both of these succeeded against 192.168.10.53:2222:

In [ ]:
PASS_FILE_PATH="../secrets/rsync.nas-01.pass"
cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"

# Test
rsync -e 'ssh -p 2222' nas-01@192.168.10.53::

CkgEazDEysPvosxz6C5SbrcS2KukQwX7XKDx6htnWhTVdKggb3s5qIpaeV3cL6Ni


In [ ]:
rsync -e 'ssh -p 2222' nas-01@192.168.10.53::nas-01

## Rsyncd Testing

In [ ]:
# List Rsync Modules
docker exec rsyncd-server rsync rsync://127.0.0.1/

In [ ]:
# List Rsync Modules
rsync rsync://nas-03.internal

In [ ]:
# List Rsync Module Contents (example)
cat ../secrets/rsync.example.pass
chmod 600 ../secrets/rsync.example.pass
rsync --password-file=../secrets/rsync.example.pass rsync://servers_user@nas-03.internal:873/servers-main/

In [ ]:
# List Rsync Module Contents (testing)
PASS_FILE_PATH="../secrets/rsync.test.pass"
# cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"
rsync --password-file=${PASS_FILE_PATH} rsync://test_user@nas-03.internal:873/testing/

In [ ]:
# Send a test file (testing)

# Password file
PASS_FILE_PATH="../secrets/rsync.test.pass"
# cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"

# File setup
TEST_FOLDER_PATH="/tmp/rsyncd-test"
mkdir --parents "${TEST_FOLDER_PATH}"
printf 'hello\n' > "${TEST_FOLDER_PATH}/hello.txt"

# Upload
rsync --archive --verbose \
  --password-file="${PASS_FILE_PATH}" \
  "${TEST_FOLDER_PATH}/" \
  "rsync://test_user@nas-03.internal:873/testing/"